# Two-Stage NIDS with Selective SMOTE — full experimental pipeline

Two-stage network intrusion detection on CSE-CIC-IDS2018: a Stage-1 binary gate (benign vs attack)
followed by a Stage-2 attack-type ensemble, with SMOTE applied selectively to the rare attack classes.

**Input:** `combined_dataset.csv`, built by `build_dataset.py` from the cleaned CSE-CIC-IDS2018 release
on Mendeley Data (Mohamed, 2024, doi:10.17632/29hdbdzx2r.1, CC BY 4.0). See `README.md`.

| Component | Setting |
|---|---|
| Data | `combined_dataset.csv` (CSE-CIC-IDS2018, 15 classes) |
| Cleaning | ±inf → NaN, drop NaN rows, drop duplicate rows, drop identifier columns if present |
| Split | 80/20 stratified, seed 42, **split before feature selection** |
| Feature selection (train only) | Information Gain (mutual information, 300k-row subsample) top-45 → Pearson pruning \|r\| > 0.90 |
| Stage 1 (gate) | LightGBM binary, Benign vs Attack, `scale_pos_weight` = benign/attack ratio |
| Stage 2 (expert) | Stacking: RF + ExtraTrees + LightGBM → XGBoost meta-learner, trained on attack rows only (14 classes) |
| Flat baselines | RF (`balanced_subsample`), same stacking ensemble on all 15 classes |
| SMOTE | classes with < 10,000 training rows raised to 25,000 |

**Experimental arms** (all evaluated on the same untouched, imbalanced 20 % test set):

| Arm | Architecture | Oversampling |
|---|---|---|
| A1 | Flat RF | none (class-weighted) |
| A2 | Flat stack | none |
| A3 | Flat stack | selective SMOTE |
| B1 | Two-stage | none |
| **B2 (proposed)** | **Two-stage** | **selective SMOTE on Stage-2 training data** |
| B3 | Two-stage | SMOTE on *all* minority attack classes (incl. the two ultra-rare ones) |

A2/A3/B1/B2 form a 2 × 2 factorial (architecture × SMOTE); B3 isolates the effect of
oversampling classes with only ~43 real training rows.

**Evaluation safeguards**
1. The split happens before feature selection; feature selection, SMOTE, threshold selection and training use the training split only. The test set is never resampled.
2. The gate threshold is chosen on out-of-fold *training* predictions. The test-set threshold sweep is a sensitivity analysis only.
3. End-to-end labels use hard gating (Stage 2 sees only flagged flows); ROC/PR-AUC use composed probabilities P(Benign)=1−p, P(c)=p·P(c|attack).
4. Stacking CV folds are shuffled (matters for SMOTE data, where synthetic rows are appended at the end).
5. Statistics: McNemar tests with Holm correction, paired bootstrap CIs, five training seeds for B1 vs B2.

**Known limitations (also stated in the paper)**
* SMOTE runs before stacking, so synthetic rows enter the stack's internal 3-fold CV; the meta-learner sees optimistic level-0 probabilities for oversampled classes. The test set is unaffected.
* Pearson pruning drops a feature correlated with *any* higher-ranked feature, including features that were themselves dropped.
* Mutual information is estimated on a 300,000-row random sample, which holds only a few rows of the rarest classes.
* SMOTE interpolates `Dst Port` as a continuous value.
* LightGBM is not bit-for-bit deterministic across thread counts and machines unless `deterministic=True`; reruns can differ in the last decimals.

**Running:** Google Colab, high-memory CPU runtime (reported run: 44 vCPU AMD EPYC 9B14, 186 GB RAM, about 1.5–2 h).
Run top to bottom. Every expensive step is checkpointed; after a disconnect, rerun all cells and finished steps load from disk.
Outside Colab, set `NIDS_DATA_PATH` and `NIDS_OUT_DIR`; the Drive cell does nothing there, and the install cell installs the versions in `requirements.txt`.
The last cell writes `results.zip` with every table and figure.

In [ ]:
# Versions used for the reported results. If pip changes numpy, pandas or scikit-learn,
# restart the runtime once (Runtime → Restart session) and run from the top again.
%pip install -q numpy==2.1.3 pandas==2.2.3 scikit-learn==1.6.1 scipy==1.16.3 lightgbm==4.7.0 xgboost==3.4.1 imbalanced-learn==0.14.2 matplotlib==3.10.0 joblib pyarrow openpyxl psutil

In [ ]:
try:                                   # mounts Google Drive in Colab; does nothing elsewhere
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab: set NIDS_DATA_PATH and NIDS_OUT_DIR before the configuration cell.")

## 0. Configuration

In [ ]:
import os, sys, json, time, platform, itertools, shutil, subprocess, hashlib

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score,
                             confusion_matrix, roc_curve, precision_recall_curve)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import binomtest, chi2

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ---------------- versions used for the reported results ----------------
EXPECTED_VERSIONS = {"numpy": "2.1.3", "pandas": "2.2.3", "sklearn": "1.6.1", "scipy": "1.16.3",
                     "lightgbm": "4.7.0", "xgboost": "3.4.1", "imblearn": "0.14.2"}
def check_versions():
    import importlib
    diff = {m: (importlib.import_module(m).__version__, v) for m, v in EXPECTED_VERSIONS.items()
            if importlib.import_module(m).__version__ != v}
    if diff:
        print("NOTE: library versions differ from the reported run (installed, reported):")
        for m, (a, b) in diff.items():
            print(f"  {m:10s} {a:10s} {b}")
        print("Results may differ slightly. See requirements.txt.")
    else:
        print("Library versions match the reported run.")
check_versions()

# ---------------- paths (override with environment variables outside Colab) ----------------
DATA_PATH = os.environ.get("NIDS_DATA_PATH", "/content/drive/MyDrive/Datasets/combined_dataset.csv")
OUT_DIR   = os.environ.get("NIDS_OUT_DIR",   "/content/drive/MyDrive/Datasets/ICRASST_SMOTE")
CKPT_DIR  = os.path.join(OUT_DIR, "checkpoints")
RES_DIR   = os.path.join(OUT_DIR, "results")
FIG_DIR   = os.path.join(RES_DIR, "figures")

LABEL_COL    = "Label"
BENIGN_LABEL = "Benign"
ID_COLS      = ["Timestamp", "Flow ID", "Src IP", "Dst IP", "Src Port"]

# ---------------- protocol ----------------
SEED           = 42
TEST_SIZE      = 0.20
IG_SAMPLE      = 300_000     # rows of the training split used to estimate mutual information
IG_TOP_K       = 45
CORR_THRESHOLD = 0.90
DEDUP_AFTER_ID_DROP = False  # True = also drop rows that become identical once identifier columns are removed

# ---------------- SMOTE ----------------
MINORITY_CUTOFF = 10_000     # classes with fewer training rows are "minority"
AUG_TARGET      = 25_000     # minority classes are raised to this many rows
SMOTE_K         = 5
SMOTE_EXCLUDE   = ["DoS attacks-SlowHTTPTest", "FTP-BruteForce"]   # ~43 real rows each: left un-augmented in A3/B2

# ---------------- Stage-1 threshold ----------------
THRESHOLD_CRITERION = "f1_attack"   # "f1_attack" (max attack-class F1) | "youden" (max TPR-FPR) | "fixed"
FIXED_THRESHOLD     = 0.5
GATE_OOF_FOLDS      = 3
THRESHOLD_GRID      = np.round(np.arange(0.05, 0.951, 0.01), 2)
SWEEP_THRESHOLDS    = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]   # sensitivity analysis only

# ---------------- what to run ----------------
RUN_FLAT_RF          = True
RUN_FLAT_STACK       = True    # longest step: stacking on ~4.1M rows
RUN_FLAT_STACK_SMOTE = True
RUN_SMOTE_ALL_ARM    = True    # B3
SEEDS_REPEAT         = [42, 7, 21, 63, 99]   # B1 vs B2 retrained per seed (gate and test set fixed); [] to skip
N_BOOT               = 1000
FORCE_RETRAIN        = False   # True ignores all checkpoints
SAVE_MODELS          = False   # full-data RF/ET stacks can reach tens of GB; predictions are always saved

# ---------------- hyperparameters ----------------
HP = {
    "rf_baseline":    dict(n_estimators=200, class_weight="balanced_subsample"),
    # The reported run also passed subsample=0.8, which LightGBM ignores without subsample_freq > 0
    # (no row bagging). Removing it gives an identical model, so it is left out here.
    "gate_lgbm":      dict(n_estimators=400, num_leaves=64, learning_rate=0.1, colsample_bytree=0.8),
    "stack_rf":       dict(n_estimators=200),
    "stack_et":       dict(n_estimators=200),
    "stack_lgbm":     dict(n_estimators=300, num_leaves=64),
    "stack_meta_xgb": dict(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist"),
    "stack_cv_folds": 3,
}

ARM_LABELS = {
    "A1": "Flat RF",
    "A2": "Flat stack",
    "A3": "Flat stack + S-SMOTE",
    "B1": "Two-stage",
    "B2": "Two-stage + S-SMOTE (proposed)",
    "B3": "Two-stage + SMOTE-all",
}
PROPOSED = "B2"

for d in (CKPT_DIR, RES_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

## 1. Helpers: checkpoints, timing, environment log

In [ ]:
def ckpt(name):  return os.path.join(CKPT_DIR, name)
def res(name):   return os.path.join(RES_DIR, name)
def fig(name):   return os.path.join(FIG_DIR, name)
def have(*names): return (not FORCE_RETRAIN) and all(os.path.exists(ckpt(n)) for n in names)

def save_json(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray):     return o.tolist()
        return str(o)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=conv)

def load_json(path):
    with open(path) as f:
        return json.load(f)

class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter(); return self
    def __exit__(self, *a):
        self.s = time.perf_counter() - self.t0

def log_environment():
    import sklearn, lightgbm, xgboost, imblearn, scipy
    env = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
        "lightgbm": lightgbm.__version__, "xgboost": xgboost.__version__,
        "imbalanced-learn": imblearn.__version__, "scipy": scipy.__version__,
        "matplotlib": matplotlib.__version__,
    }
    for mod in ("pyarrow", "openpyxl", "joblib"):
        try:
            env[mod] = __import__(mod).__version__
        except ImportError:
            pass
    try:
        import psutil
        env["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except ImportError:
        pass
    try:
        env["cpu_model"] = subprocess.run("lscpu | grep 'Model name'", shell=True,
                                          capture_output=True, text=True).stdout.split(":")[-1].strip()
    except Exception:
        pass
    env["accelerator"] = ("TPU" if os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME")
                          else "GPU" if shutil.which("nvidia-smi") else "CPU only")
    env["note"] = "All models in this pipeline run on CPU."
    env["config"] = {k: globals()[k] for k in [
        "SEED", "TEST_SIZE", "IG_SAMPLE", "IG_TOP_K", "CORR_THRESHOLD", "DEDUP_AFTER_ID_DROP",
        "MINORITY_CUTOFF", "AUG_TARGET", "SMOTE_K", "SMOTE_EXCLUDE", "THRESHOLD_CRITERION",
        "GATE_OOF_FOLDS", "SEEDS_REPEAT", "N_BOOT"]}
    env["hyperparameters"] = HP
    save_json(env, res("environment.json"))
    for k, v in env.items():
        if k not in ("config", "hyperparameters"):
            print(f"  {k:18s}: {v}")
    return env

ENV = log_environment()

## 2. Preprocessing and feature selection
Cleaning, split and feature selection. Also records the cleaning counts, the identifier-free duplicate
diagnostic and the train/test overlap needed for the dataset and threats-to-validity sections.

In [ ]:
def stage1_preprocess():
    files = ["X_train.parquet", "X_test.parquet", "y_train.npy", "y_test.npy",
             "label_encoder.joblib", "prep_log.json"]
    if have(*files):
        print("[prep] checkpoints found — loading")
        return (pd.read_parquet(ckpt("X_train.parquet")), pd.read_parquet(ckpt("X_test.parquet")),
                np.load(ckpt("y_train.npy")), np.load(ckpt("y_test.npy")),
                joblib.load(ckpt("label_encoder.joblib")), load_json(ckpt("prep_log.json")))

    log = {}
    print("[prep] loading", DATA_PATH)
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df.columns = df.columns.str.strip()
    log["rows_raw"], log["cols_raw"] = int(len(df)), int(df.shape[1])
    log["input_fingerprint"] = hashlib.sha256(
        pd.util.hash_pandas_object(df, index=False).values.tobytes()).hexdigest()   # row order matters

    hdr = df[LABEL_COL].astype(str) == LABEL_COL            # repeated CSV header rows, if any
    df = df[~hdr]
    log["header_rows_removed"] = int(hdr.sum())

    df = df.replace([np.inf, -np.inf], np.nan)
    n0 = len(df); df = df.dropna();          log["rows_removed_nan_inf"] = int(n0 - len(df))
    n0 = len(df); df = df.drop_duplicates(); log["rows_removed_duplicates"] = int(n0 - len(df))

    present_ids = [c for c in ID_COLS if c in df.columns]
    df = df.drop(columns=present_ids)
    log["identifier_columns_dropped"] = present_ids

    # duplicates that only appear once identifiers are gone
    log["dup_rows_after_id_drop"] = int(df.duplicated().sum())
    feat_cols = [c for c in df.columns if c != LABEL_COL]
    h = pd.util.hash_pandas_object(df[feat_cols], index=False).values
    lab_per_vec = pd.Series(df[LABEL_COL].values).groupby(h).nunique()
    log["feature_vectors_with_conflicting_labels"] = int((lab_per_vec > 1).sum())
    if DEDUP_AFTER_ID_DROP:
        n0 = len(df); df = df.drop_duplicates()
        log["rows_removed_dedup_after_id_drop"] = int(n0 - len(df))
    log["rows_clean"] = int(len(df))

    le = LabelEncoder()
    y = le.fit_transform(df[LABEL_COL])
    X_all = df.drop(columns=[LABEL_COL])
    non_numeric = [c for c in X_all.columns if not pd.api.types.is_numeric_dtype(X_all[c])]
    X = X_all.select_dtypes(include=[np.number])
    log["non_numeric_columns_dropped"] = non_numeric
    log["n_numeric_features"] = int(X.shape[1])
    del df, X_all
    joblib.dump(le, ckpt("label_encoder.joblib"))

    # split FIRST, then select features on the training split only
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
    log["rows_train"], log["rows_test"] = int(len(X_train)), int(len(X_test))

    print("[prep] information gain (mutual information) ...")
    n_ig = min(len(X_train), IG_SAMPLE)
    idx = np.random.RandomState(SEED).choice(len(X_train), n_ig, replace=False)
    mi = mutual_info_classif(X_train.iloc[idx], y_train[idx], random_state=SEED,
                             discrete_features=False)
    ig = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
    top = ig.head(IG_TOP_K).index.tolist()

    # Pearson pruning: a feature is dropped if |r| > threshold with
    # ANY higher-IG feature in the top-K list
    corr = X_train[top].corr(method="pearson").abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    dropped_by = {}
    for c in upper.columns:
        hit = upper[c][upper[c] > CORR_THRESHOLD]
        if len(hit):
            dropped_by[c] = f"{hit.index[0]} (|r|={hit.iloc[0]:.3f})"
    selected = [f for f in top if f not in dropped_by]
    log["ig_rows_used"] = int(n_ig)
    log["n_selected_features"] = len(selected)
    print(f"[prep] top-{IG_TOP_K} by IG -> {len(dropped_by)} dropped by correlation -> "
          f"{len(selected)} features")

    feat_tab = pd.DataFrame({"feature": ig.index, "ig_score": ig.values,
                             "ig_rank": np.arange(1, len(ig) + 1)})
    feat_tab["in_top_k"] = feat_tab["feature"].isin(top)
    feat_tab["selected"] = feat_tab["feature"].isin(selected)
    feat_tab["dropped_correlated_with"] = feat_tab["feature"].map(dropped_by).fillna("")
    feat_tab.to_csv(res("table_feature_selection.csv"), index=False)
    save_json(selected, ckpt("selected_features.json"))

    X_train = X_train[selected].reset_index(drop=True)
    X_test  = X_test[selected].reset_index(drop=True)

    # train/test overlap in the selected feature space (threats-to-validity diagnostic)
    h_tr = pd.util.hash_pandas_object(X_train, index=False).values
    h_te = pd.util.hash_pandas_object(X_test,  index=False).values
    in_train = np.isin(h_te, h_tr)
    log["test_rows_with_identical_train_vector"] = int(in_train.sum())
    log["test_overlap_fraction"] = float(in_train.mean())
    ov = pd.DataFrame({"class": le.inverse_transform(y_test), "in_train": in_train}) \
           .groupby("class")["in_train"].agg(["sum", "mean", "size"]) \
           .rename(columns={"sum": "test_rows_seen_in_train", "mean": "fraction", "size": "test_rows"})
    ov.to_csv(res("table_train_test_overlap.csv"))

    X_train.to_parquet(ckpt("X_train.parquet")); X_test.to_parquet(ckpt("X_test.parquet"))
    np.save(ckpt("y_train.npy"), y_train);       np.save(ckpt("y_test.npy"), y_test)
    save_json(log, ckpt("prep_log.json")); save_json(log, res("preprocessing_log.json"))
    return X_train, X_test, y_train, y_test, le, log


X_train, X_test, y_train, y_test, le, PREP_LOG = stage1_preprocess()
CLASS_NAMES = list(le.classes_)
K           = len(CLASS_NAMES)
BENIGN      = CLASS_NAMES.index(BENIGN_LABEL)
ATTACK_CODES = np.array([c for c in range(K) if c != BENIGN])
ATTACK_NAMES = [CLASS_NAMES[c] for c in ATTACK_CODES]
TO_COMPACT   = {int(c): i for i, c in enumerate(ATTACK_CODES)}

train_counts = pd.Series(y_train).value_counts().reindex(range(K), fill_value=0)
MINORITY_NAMES = [CLASS_NAMES[c] for c in range(K) if train_counts[c] < MINORITY_CUTOFF]
MINORITY_FULL  = np.array([CLASS_NAMES.index(n) for n in MINORITY_NAMES])
MINORITY_COMPACT = np.array([TO_COMPACT[c] for c in MINORITY_FULL])

for k, v in PREP_LOG.items():
    print(f"  {k:42s}: {v}")
print("Minority classes (<%d training rows): %s" % (MINORITY_CUTOFF, MINORITY_NAMES))

## 3. SMOTE
Oversampling touches **training data only**. Stage-1 is always trained on the original data;
SMOTE is applied to the attack-only Stage-2 training set (B2, B3) or the full 15-class
training set (A3). The test set is never resampled.

In [ ]:
def smote_augment(X, y, names, exclude, seed):
    """Raise every class with < MINORITY_CUTOFF rows to AUG_TARGET, except `exclude`.
    `names[i]` is the class name of label i in this label space."""
    assert AUG_TARGET >= MINORITY_CUTOFF
    counts = pd.Series(y).value_counts()
    strategy = {int(c): AUG_TARGET for c, n in counts.items()
                if n < MINORITY_CUTOFF and names[int(c)] not in exclude}
    if not strategy:
        return X, y
    k = max(1, min(SMOTE_K, min(counts[c] for c in strategy) - 1))
    sm = SMOTE(sampling_strategy=strategy, k_neighbors=k, random_state=seed)
    Xr, yr = sm.fit_resample(X, y)
    return pd.DataFrame(np.asarray(Xr), columns=X.columns), np.asarray(yr)


# Stage-2 training data: attack rows only, compact labels 0..13
att_mask_tr = y_train != BENIGN
X_att = X_train[att_mask_tr].reset_index(drop=True)
y_att = np.array([TO_COMPACT[int(c)] for c in y_train[att_mask_tr]])

SMOTE_VARIANTS = {   # arm -> (excluded classes)
    "B2": SMOTE_EXCLUDE,
    "B3": [],
}

# class-distribution table for the paper (full / train / test / Stage-2 training after each SMOTE variant)
dist = pd.DataFrame({
    "class": CLASS_NAMES,
    "full":  pd.Series(y_train).value_counts().add(pd.Series(y_test).value_counts(), fill_value=0)
              .reindex(range(K), fill_value=0).astype(int).values,
    "train": train_counts.values,
    "test":  pd.Series(y_test).value_counts().reindex(range(K), fill_value=0).values,
})
dist["minority"] = dist["class"].isin(MINORITY_NAMES)
for arm, excl in SMOTE_VARIANTS.items():
    _, y_tmp = smote_augment(X_att, y_att, ATTACK_NAMES, excl, SEED)
    cnt = pd.Series(y_tmp).value_counts()
    dist[f"stage2_train_{arm}"] = [np.nan if c == BENIGN else int(cnt.get(TO_COMPACT[c], 0))
                                   for c in range(K)]
dist = dist.sort_values("train", ascending=False)
dist.to_csv(res("table_class_distribution.csv"), index=False)
print(dist.to_string(index=False))

## 4. Model builders and metrics

In [ ]:
def build_rf_baseline(seed=SEED):
    return RandomForestClassifier(**HP["rf_baseline"], n_jobs=-1, random_state=seed)

def build_gate(scale_pos_weight, seed=SEED):
    return LGBMClassifier(**HP["gate_lgbm"], scale_pos_weight=scale_pos_weight,
                          random_state=seed, n_jobs=-1, verbose=-1)

def build_stack(n_classes, seed=SEED):
    level0 = [
        ("rf",   RandomForestClassifier(**HP["stack_rf"], n_jobs=-1, random_state=seed)),
        ("et",   ExtraTreesClassifier(**HP["stack_et"], n_jobs=-1, random_state=seed)),
        ("lgbm", LGBMClassifier(**HP["stack_lgbm"], random_state=seed, n_jobs=-1, verbose=-1)),
    ]
    meta = XGBClassifier(**HP["stack_meta_xgb"], objective="multi:softprob", num_class=n_classes,
                         random_state=seed, n_jobs=-1, eval_metric="mlogloss")
    cv = StratifiedKFold(n_splits=HP["stack_cv_folds"], shuffle=True, random_state=seed)
    return StackingClassifier(estimators=level0, final_estimator=meta,
                              stack_method="predict_proba", cv=cv, n_jobs=1, passthrough=False)


def compute_metrics(y_true, y_pred, proba, names, minority, tag, benign=None):
    """Full metric suite. `minority` = label indices of minority classes in this label space.
    `benign` = index of the benign class (end-to-end / flat evaluation only)."""
    labels = np.arange(len(names))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    m = {
        "tag": tag,
        "accuracy":           accuracy_score(y_true, y_pred),
        "macro_precision":    p.mean(),
        "macro_recall":       r.mean(),            # = balanced accuracy
        "macro_f1":           f.mean(),
        "weighted_f1":        f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "minority_macro_f1":  f[minority].mean(),
        "mcc":                matthews_corrcoef(y_true, y_pred),
    }
    if benign is not None:
        is_b, pred_b = y_true == benign, y_pred == benign
        m["attack_detection_rate"] = float((~pred_b[~is_b]).mean())   # attacks flagged as any attack
        m["benign_fpr"]            = float((~pred_b[is_b]).mean())    # benign flagged as attack
    if proba is not None:
        P = np.asarray(proba, dtype=np.float64)
        P = P / P.sum(axis=1, keepdims=True)
        try:
            m["macro_roc_auc"] = roc_auc_score(y_true, P, multi_class="ovr", average="macro", labels=labels)
            Y = label_binarize(y_true, classes=labels)
            m["macro_pr_auc"]  = average_precision_score(Y, P, average="macro")
        except ValueError as e:
            print("  AUC skipped:", e)
    per_class = pd.DataFrame({"class": names, "precision": p, "recall": r, "f1": f, "support": s})
    cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=names, columns=names)
    return m, per_class, cm


def report(m, per_class=None):
    print(f"\n===== {m['tag']} =====")
    for k, v in m.items():
        if k != "tag":
            print(f"  {k:24s}: {v:.4f}" if isinstance(v, (float, np.floating)) else f"  {k:24s}: {v}")
    if per_class is not None:
        print(per_class.round(4).to_string(index=False))


RESULTS = {}     # arm -> dict(pred, proba, metrics, per_class, cm, timing)

def store(arm, y_pred, proba, timing):
    m, pc, cm = compute_metrics(y_test, y_pred, proba, CLASS_NAMES, MINORITY_FULL,
                                f"{arm}: {ARM_LABELS[arm]}", benign=BENIGN)
    m.update(timing)
    RESULTS[arm] = dict(pred=y_pred, proba=proba, metrics=m, per_class=pc, cm=cm)
    pc.to_csv(res(f"per_class_{arm}.csv"), index=False)
    cm.to_csv(res(f"confusion_matrix_{arm}.csv"))
    save_json(m, res(f"metrics_{arm}.json"))
    report(m, pc)

## 5. Flat baselines (A1, A2, A3)

In [ ]:
def run_flat_arm(arm, build, X_fit, y_fit):
    pred_f, proba_f, time_f = f"pred_{arm}.npy", f"proba_{arm}.npy", f"timing_{arm}.json"
    if have(pred_f, proba_f, time_f):
        print(f"[{arm}] predictions found — loading")
        store(arm, np.load(ckpt(pred_f)), np.load(ckpt(proba_f)), load_json(ckpt(time_f)))
        return
    print(f"[{arm}] training {ARM_LABELS[arm]} on {len(X_fit):,} rows ...")
    model = build()
    with Timer() as t_fit:
        model.fit(X_fit, y_fit)
    assert list(model.classes_) == list(range(K))
    with Timer() as t_pred:
        proba = model.predict_proba(X_test).astype(np.float32)
    pred = proba.argmax(axis=1)
    timing = {"train_rows": int(len(X_fit)), "train_seconds": t_fit.s,
              "test_predict_seconds": t_pred.s,
              "batch_inference_us_per_flow": 1e6 * t_pred.s / len(X_test)}
    np.save(ckpt(pred_f), pred); np.save(ckpt(proba_f), proba); save_json(timing, ckpt(time_f))
    if SAVE_MODELS:
        joblib.dump(model, ckpt(f"model_{arm}.joblib"))
    del model
    store(arm, pred, proba, timing)


if RUN_FLAT_RF:
    run_flat_arm("A1", build_rf_baseline, X_train, y_train)
if RUN_FLAT_STACK:
    run_flat_arm("A2", lambda: build_stack(K), X_train, y_train)
if RUN_FLAT_STACK_SMOTE:
    if have("pred_A3.npy", "proba_A3.npy", "timing_A3.json"):
        run_flat_arm("A3", None, None, None)
    else:
        with Timer() as t_sm:
            X_a3, y_a3 = smote_augment(X_train, y_train, CLASS_NAMES, SMOTE_EXCLUDE, SEED)
        print(f"[A3] SMOTE {len(X_train):,} -> {len(X_a3):,} rows in {t_sm.s:.0f}s")
        run_flat_arm("A3", lambda: build_stack(K), X_a3, y_a3)
        del X_a3, y_a3

## 6. Stage 1 — binary gate (Benign vs Attack)
Trained on the original (non-oversampled) training data. The decision threshold is chosen
on 3-fold out-of-fold predictions over the training split.

In [ ]:
def choose_threshold(y_bin, p):
    rows = []
    P1 = y_bin == 1
    for t in THRESHOLD_GRID:
        pr = p >= t
        tp = int((pr & P1).sum()); fp = int((pr & ~P1).sum())
        fn = int((~pr & P1).sum()); tn = int((~pr & ~P1).sum())
        rows.append({"threshold": t, "f1_attack": 2 * tp / max(2 * tp + fp + fn, 1),
                     "tpr": tp / max(tp + fn, 1), "fpr": fp / max(fp + tn, 1)})
    grid = pd.DataFrame(rows)
    grid["youden"] = grid["tpr"] - grid["fpr"]
    if THRESHOLD_CRITERION == "fixed":
        return FIXED_THRESHOLD, grid
    return float(grid.loc[grid[THRESHOLD_CRITERION].idxmax(), "threshold"]), grid


def gate_binary_metrics(y_bin, p, thr, tag):
    pred = (p >= thr).astype(int)
    p_, r_, f_, s_ = precision_recall_fscore_support(y_bin, pred, labels=[0, 1], zero_division=0)
    cm = confusion_matrix(y_bin, pred, labels=[0, 1])
    m = {"tag": tag, "threshold": thr, "accuracy": accuracy_score(y_bin, pred),
         "precision_benign": p_[0], "recall_benign": r_[0], "f1_benign": f_[0],
         "precision_attack": p_[1], "recall_attack": r_[1], "f1_attack": f_[1],
         "fpr": cm[0, 1] / cm[0].sum(), "fnr": cm[1, 0] / cm[1].sum(),
         "roc_auc": roc_auc_score(y_bin, p), "pr_auc": average_precision_score(y_bin, p),
         "tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1])}
    return m


ybin_tr = (y_train != BENIGN).astype(int)
ybin_te = (y_test != BENIGN).astype(int)
SPW = float((ybin_tr == 0).sum() / (ybin_tr == 1).sum())
print(f"scale_pos_weight = benign/attack = {SPW:.3f}")

if have("gate.joblib", "gate_oof.npy", "gate_test_proba.npy", "gate_timing.json"):
    print("[gate] checkpoints found — loading")
    gate = joblib.load(ckpt("gate.joblib"))
    p_oof = np.load(ckpt("gate_oof.npy")); p_gate = np.load(ckpt("gate_test_proba.npy"))
    GATE_TIMING = load_json(ckpt("gate_timing.json"))
else:
    cv = StratifiedKFold(GATE_OOF_FOLDS, shuffle=True, random_state=SEED)
    print("[gate] out-of-fold predictions for threshold selection ...")
    p_oof = cross_val_predict(build_gate(SPW), X_train, ybin_tr, cv=cv,
                              method="predict_proba", n_jobs=1)[:, 1]
    gate = build_gate(SPW)
    with Timer() as t_fit:
        gate.fit(X_train, ybin_tr)
    with Timer() as t_pred:
        p_gate = gate.predict_proba(X_test)[:, 1]
    GATE_TIMING = {"train_seconds": t_fit.s, "test_predict_seconds": t_pred.s}
    joblib.dump(gate, ckpt("gate.joblib"))
    np.save(ckpt("gate_oof.npy"), p_oof); np.save(ckpt("gate_test_proba.npy"), p_gate)
    save_json(GATE_TIMING, ckpt("gate_timing.json"))

THR, thr_grid = choose_threshold(ybin_tr, p_oof)
thr_grid.to_csv(res("gate_threshold_selection_oof.csv"), index=False)
print(f"[gate] threshold chosen on training OOF ({THRESHOLD_CRITERION}): {THR:.2f}")

gate_rows = [gate_binary_metrics(ybin_tr, p_oof, THR, "train OOF @ chosen"),
             gate_binary_metrics(ybin_te, p_gate, THR, "test @ chosen"),
             gate_binary_metrics(ybin_te, p_gate, 0.5, "test @ 0.50")]
GATE_TABLE = pd.DataFrame(gate_rows).set_index("tag")
GATE_TABLE.to_csv(res("table_gate_binary.csv"))
print(GATE_TABLE.T.round(4).to_string())

## 7. Stage 2 — attack-type stack (B1, B2, B3) and end-to-end evaluation
Stage 2 is trained on attack rows only. Its class probabilities are computed for every test row
once (so thresholds can be swept without refitting); end-to-end labels use hard gating:
rows with P(attack) < τ → Benign, otherwise the Stage-2 argmax.

In [ ]:
def compose(p_att, cond, thr):
    """Hard-gated labels + composed probabilities in the full 15-class space."""
    pred = np.full(len(p_att), BENIGN, dtype=np.int64)
    flag = p_att >= thr
    pred[flag] = ATTACK_CODES[cond[flag].argmax(axis=1)]
    proba = np.zeros((len(p_att), K), dtype=np.float32)
    proba[:, BENIGN] = 1.0 - p_att
    proba[:, ATTACK_CODES] = p_att[:, None] * cond
    return pred, proba


def fit_stage2(arm, seed):
    """Train the Stage-2 stack for one arm/seed; return conditional test probabilities + timing."""
    tag = f"{arm}_s{seed}"
    cond_f, time_f = f"cond_{tag}.npy", f"timing_stage2_{tag}.json"
    if have(cond_f, time_f):
        return np.load(ckpt(cond_f)), load_json(ckpt(time_f))
    timing = {}
    if arm == "B1":
        Xf, yf = X_att, y_att
    else:
        with Timer() as t_sm:
            Xf, yf = smote_augment(X_att, y_att, ATTACK_NAMES, SMOTE_VARIANTS[arm], seed)
        timing["smote_seconds"] = t_sm.s
    print(f"[{tag}] training Stage-2 stack on {len(Xf):,} attack rows ...")
    stack = build_stack(len(ATTACK_CODES), seed)
    with Timer() as t_fit:
        stack.fit(Xf, yf)
    with Timer() as t_all:
        cond = stack.predict_proba(X_test).astype(np.float32)
    flag = p_gate >= THR
    with Timer() as t_flag:                       # deployment-faithful: Stage 2 sees flagged rows only
        stack.predict_proba(X_test[flag])
    timing.update({"stage2_train_rows": int(len(Xf)), "stage2_train_seconds": t_fit.s,
                   "stage2_predict_all_seconds": t_all.s,
                   "stage2_predict_flagged_seconds": t_flag.s, "flagged_rows": int(flag.sum())})
    np.save(ckpt(cond_f), cond); save_json(timing, ckpt(time_f))
    if SAVE_MODELS and seed == SEED:
        joblib.dump(stack, ckpt(f"model_stage2_{arm}.joblib"))
    del stack
    return cond, timing


ISOLATED = {}
COND = {}
two_stage_arms = ["B1", "B2"] + (["B3"] if RUN_SMOTE_ALL_ARM else [])
att_mask_te = y_test != BENIGN
y_att_te = np.array([TO_COMPACT[int(c)] for c in y_test[att_mask_te]])

for arm in two_stage_arms:
    cond, t2 = fit_stage2(arm, SEED)
    COND[arm] = cond
    # (i) Stage 2 in isolation: true attack rows only, no gate errors
    mi, pci, _ = compute_metrics(y_att_te, cond[att_mask_te].argmax(1), cond[att_mask_te],
                                 ATTACK_NAMES, MINORITY_COMPACT, f"{arm} Stage-2 isolated")
    ISOLATED[arm] = (mi, pci)
    pci.to_csv(res(f"per_class_stage2_isolated_{arm}.csv"), index=False)
    report(mi)
    # (ii) end-to-end at the OOF-selected threshold
    pred, proba = compose(p_gate, cond, THR)
    timing = {"threshold": THR,
              "train_seconds": GATE_TIMING["train_seconds"] + t2.get("smote_seconds", 0)
                               + t2["stage2_train_seconds"],
              "test_predict_seconds": GATE_TIMING["test_predict_seconds"]
                                      + t2["stage2_predict_flagged_seconds"],
              **t2}
    timing["batch_inference_us_per_flow"] = 1e6 * timing["test_predict_seconds"] / len(X_test)
    store(arm, pred, proba, timing)

pd.DataFrame({a: ISOLATED[a][0] for a in ISOLATED}).T.set_index("tag") \
  .to_csv(res("table_stage2_isolated.csv"))

## 8. Ablation summary and per-class tables

In [ ]:
ARMS = [a for a in ["A1", "A2", "A3", "B1", "B2", "B3"] if a in RESULTS]
cols = ["macro_f1", "minority_macro_f1", "weighted_f1", "accuracy", "macro_precision",
        "macro_recall", "mcc", "macro_roc_auc", "macro_pr_auc", "attack_detection_rate",
        "benign_fpr", "train_seconds", "batch_inference_us_per_flow"]
ABL = pd.DataFrame({a: RESULTS[a]["metrics"] for a in ARMS}).T
ABL = ABL[[c for c in cols if c in ABL.columns]].astype(float)
ABL.insert(0, "configuration", [ARM_LABELS[a] for a in ARMS])
ABL.to_csv(res("table_ablation.csv"), index_label="arm")
print(ABL.round(4).to_string())

PER_CLASS_F1 = pd.DataFrame({a: RESULTS[a]["per_class"].set_index("class")["f1"] for a in ARMS})
PER_CLASS_F1.insert(0, "train_rows", train_counts.values)
PER_CLASS_F1.insert(1, "test_rows", RESULTS[ARMS[0]]["per_class"]["support"].values)
PER_CLASS_F1 = PER_CLASS_F1.sort_values("train_rows", ascending=False)
PER_CLASS_F1.to_csv(res("table_per_class_f1_all_arms.csv"), index_label="class")
print(PER_CLASS_F1.round(3).to_string())

## 9. Statistical comparison
* **McNemar** (proposed vs each other arm) on per-sample correctness — all test rows, and rows of
  minority classes only. Exact binomial test when discordant pairs < 25, otherwise χ² with continuity
  correction. Holm–Bonferroni correction within each subset.
* **Bootstrap** (N_BOOT resamples of the test set): 95 % percentile CI for each arm's macro-F1 and
  minority macro-F1, and paired CIs for (proposed − other).

With ~1M test rows McNemar flags very small differences as significant — report the discordant
counts b, c next to every p-value.

In [ ]:
def mcnemar(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))     # a right, b wrong
    c = int(np.sum(~correct_a & correct_b))     # a wrong, b right
    if b + c == 0:
        return b, c, np.nan, 1.0, "no discordant pairs"
    if b + c < 25:
        return b, c, np.nan, binomtest(min(b, c), b + c, 0.5).pvalue, "exact binomial"
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    return b, c, stat, chi2.sf(stat, 1), "chi2 (continuity corrected)"

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for i, j in enumerate(order):
        run = max(run, (m - i) * p[j]); adj[j] = min(1.0, run)
    return adj

minority_rows = np.isin(y_test, MINORITY_FULL)
rows = []
for subset, mask in [("all test rows", np.ones(len(y_test), bool)),
                     ("minority-class rows", minority_rows)]:
    ok_p = RESULTS[PROPOSED]["pred"][mask] == y_test[mask]
    sub = []
    for a in ARMS:
        if a == PROPOSED: continue
        ok_a = RESULTS[a]["pred"][mask] == y_test[mask]
        b, c, stat, p, method = mcnemar(ok_p, ok_a)
        sub.append({"subset": subset, "n": int(mask.sum()), "comparison": f"{PROPOSED} vs {a}",
                    "b_proposed_right_other_wrong": b, "c_proposed_wrong_other_right": c,
                    "statistic": stat, "p_value": p, "test": method})
    adj = holm([r["p_value"] for r in sub])
    for r, pa in zip(sub, adj):
        r["p_holm"] = pa
    rows += sub
MCNEMAR = pd.DataFrame(rows)
MCNEMAR.to_csv(res("table_mcnemar.csv"), index=False)
print(MCNEMAR.to_string(index=False))


def f1_vector(yt, yp):
    cm = np.bincount(yt * K + yp, minlength=K * K).reshape(K, K)
    tp = np.diag(cm).astype(float); denom = cm.sum(0) + cm.sum(1)
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0), denom > 0

# sanity check against scikit-learn
_f, _pres = f1_vector(y_test, RESULTS[PROPOSED]["pred"])
assert abs(_f[_pres].mean() - RESULTS[PROPOSED]["metrics"]["macro_f1"]) < 1e-9

rng = np.random.default_rng(SEED)
n = len(y_test)
boot = {a: {"macro": np.empty(N_BOOT), "minority": np.empty(N_BOOT)} for a in ARMS}
for i in range(N_BOOT):
    idx = rng.integers(0, n, n)
    yt = y_test[idx]
    for a in ARMS:
        f, pres = f1_vector(yt, RESULTS[a]["pred"][idx])
        boot[a]["macro"][i] = f[pres].mean()
        boot[a]["minority"][i] = f[MINORITY_FULL].mean()

ci_rows = []
for a in ARMS:
    for kind in ("macro", "minority"):
        v = boot[a][kind]
        key = "macro_f1" if kind == "macro" else "minority_macro_f1"
        row = {"arm": a, "metric": key, "point": RESULTS[a]["metrics"][key],
               "ci_low": np.percentile(v, 2.5), "ci_high": np.percentile(v, 97.5)}
        if a != PROPOSED:
            d = boot[PROPOSED][kind] - v
            row.update({"diff_proposed_minus_arm": RESULTS[PROPOSED]["metrics"][key] - row["point"],
                        "diff_ci_low": np.percentile(d, 2.5), "diff_ci_high": np.percentile(d, 97.5),
                        "share_resamples_proposed_better": float((d > 0).mean())})
        ci_rows.append(row)
BOOT = pd.DataFrame(ci_rows)
BOOT.to_csv(res("table_bootstrap_ci.csv"), index=False)
print(BOOT.round(4).to_string(index=False))

## 10. Threshold sensitivity (test set — reported as sensitivity analysis, not used for selection)

In [ ]:
sweep = []
for arm in two_stage_arms:
    for t in sorted(set(SWEEP_THRESHOLDS) | {THR}):
        pred, _ = compose(p_gate, COND[arm], t)
        _, _, f, _ = precision_recall_fscore_support(y_test, pred, labels=np.arange(K), zero_division=0)
        is_b = y_test == BENIGN
        sweep.append({"arm": arm, "threshold": t, "selected": t == THR,
                      "macro_f1": f.mean(), "minority_macro_f1": f[MINORITY_FULL].mean(),
                      "infilteration_f1": f[CLASS_NAMES.index("Infilteration")]
                                          if "Infilteration" in CLASS_NAMES else np.nan,
                      "attack_detection_rate": float((pred[~is_b] != BENIGN).mean()),
                      "benign_fpr": float((pred[is_b] != BENIGN).mean())})
SWEEP = pd.DataFrame(sweep)
SWEEP.to_csv(res("table_threshold_sensitivity.csv"), index=False)
print(SWEEP.round(4).to_string(index=False))

## 11. Seed repeats (B1 vs B2)
Stage 2 (and SMOTE) retrained with different seeds; split, features, gate and threshold fixed, so the
spread reflects training randomness only. Each seed is checkpointed.

In [ ]:
seed_rows = []
for s in SEEDS_REPEAT:
    for arm in ["B1", "B2"]:
        cond, _ = fit_stage2(arm, s) if s != SEED else (COND[arm], None)
        pred, _ = compose(p_gate, cond, THR)
        _, _, f, _ = precision_recall_fscore_support(y_test, pred, labels=np.arange(K), zero_division=0)
        seed_rows.append({"seed": s, "arm": arm, "macro_f1": f.mean(),
                          "minority_macro_f1": f[MINORITY_FULL].mean(),
                          "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0),
                          **{f"f1_{CLASS_NAMES[c]}": f[c] for c in MINORITY_FULL}})
if seed_rows:
    SEEDS = pd.DataFrame(seed_rows)
    SEEDS.to_csv(res("table_seed_repeats.csv"), index=False)
    summ = SEEDS.groupby("arm")[["macro_f1", "minority_macro_f1", "weighted_f1"]].agg(["mean", "std"])
    summ.columns = [f"{m}_{s}" for m, s in summ.columns]          # flat, one-row header
    piv = SEEDS.pivot(index="seed", columns="arm", values="macro_f1")
    summ.to_csv(res("table_seed_summary.csv"))
    print(summ.round(4).to_string())
    print(f"B2 > B1 (macro-F1) in {(piv['B2'] > piv['B1']).sum()} of {len(piv)} seeds; "
          f"mean paired difference {(piv['B2'] - piv['B1']).mean():+.4f} "
          f"(sd {(piv['B2'] - piv['B1']).std():.4f})")

## 12. Figures (300 dpi PNG, sized for a 6.5-inch text width)

In [ ]:
plt.rcParams.update({
    "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.edgecolor": "#52514e",
    "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e5e4e0", "grid.linewidth": 0.6, "savefig.dpi": 300, "savefig.bbox": "tight",
})
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]   # fixed order
BLUES  = LinearSegmentedColormap.from_list("seq", ["#ffffff", "#cde2fb", "#86b6ef", "#3987e5",
                                                   "#1c5cab", "#0d366b"])

def savefig(f, name):
    f.savefig(fig(name + ".png")); plt.close(f); print("saved", name)

# Fig A — selected features by information gain
ft = pd.read_csv(res("table_feature_selection.csv"))
ft = ft[ft["selected"]].sort_values("ig_score")
f, ax = plt.subplots(figsize=(6.0, 0.17 * len(ft) + 0.6))
ax.barh(ft["feature"], ft["ig_score"], color=SERIES[0], height=0.7)
ax.set_xlabel("Information gain (mutual information, nats)")
ax.grid(axis="y", visible=False)
savefig(f, "fig_feature_information_gain")

# Fig B — Stage-2 training distribution before / after selective SMOTE
d2 = dist[dist["class"] != BENIGN_LABEL].sort_values("train")
f, ax = plt.subplots(figsize=(6.0, 3.6))
yy = np.arange(len(d2)); h = 0.38
ax.barh(yy + h / 2, d2["train"], h, color=SERIES[0], label="Original")
ax.barh(yy - h / 2, d2["stage2_train_B2"], h, color=SERIES[1], label="After selective SMOTE")
ax.set_yticks(yy, d2["class"]); ax.set_xscale("log"); ax.grid(axis="y", visible=False)
ax.set_xlabel("Training rows (log scale)"); ax.legend(loc="lower right", frameon=False)
savefig(f, "fig_stage2_class_distribution")

# Fig C — gate ROC and precision-recall curves with the chosen operating point
f, (a1, a2) = plt.subplots(1, 2, figsize=(6.0, 2.7))
fpr, tpr, thr_r = roc_curve(ybin_te, p_gate)
a1.plot(fpr, tpr, color=SERIES[0], lw=1.5)
g = GATE_TABLE.loc["test @ chosen"]
a1.plot(g["fpr"], g["recall_attack"], "o", ms=6, color=SERIES[1], mec="white", mew=1.5,
        label=f"τ = {THR:.2f}")
a1.set_xlabel("False positive rate"); a1.set_ylabel("True positive rate")
a1.set_title(f"ROC (AUC = {g['roc_auc']:.4f})"); a1.legend(frameon=False, loc="lower right")
prec, rec, _ = precision_recall_curve(ybin_te, p_gate)
a2.plot(rec, prec, color=SERIES[0], lw=1.5)
a2.plot(g["recall_attack"], g["precision_attack"], "o", ms=6, color=SERIES[1], mec="white", mew=1.5)
a2.set_xlabel("Recall (attack)"); a2.set_ylabel("Precision (attack)")
a2.set_title(f"Precision–recall (AP = {g['pr_auc']:.4f})")
savefig(f, "fig_gate_roc_pr")

# Fig D — per-class F1, two-stage arms
show = [a for a in two_stage_arms]
pcf = PER_CLASS_F1.iloc[::-1]
f, ax = plt.subplots(figsize=(6.0, 4.6))
yy = np.arange(len(pcf)); h = 0.8 / len(show)
for i, a in enumerate(show):
    ax.barh(yy - (i - (len(show) - 1) / 2) * h, pcf[a], h * 0.92, color=SERIES[i], label=ARM_LABELS[a])
ax.set_yticks(yy, [f"{c} ({n:,})" for c, n in zip(pcf.index, pcf["train_rows"])])
ax.set_xlabel("F1-score (test set)"); ax.set_xlim(0, 1.0); ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=len(show))
savefig(f, "fig_per_class_f1_two_stage")

# Fig E — row-normalised confusion matrix, proposed model
cm = RESULTS[PROPOSED]["cm"].values.astype(float)
cmn = cm / cm.sum(axis=1, keepdims=True)
f, ax = plt.subplots(figsize=(6.0, 5.4))
im = ax.imshow(cmn, cmap=BLUES, vmin=0, vmax=1)
ax.set_xticks(range(K), CLASS_NAMES, rotation=60, ha="right"); ax.set_yticks(range(K), CLASS_NAMES)
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class"); ax.grid(False)
for i in range(K):
    for j in range(K):
        if cmn[i, j] >= 0.01:
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=5.5,
                    color="white" if cmn[i, j] > 0.6 else "#0b0b0b")
f.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label="Row-normalised rate (recall)")
savefig(f, "fig_confusion_matrix_proposed")

# Fig F — threshold sensitivity (two panels, one scale each)
f, (a1, a2) = plt.subplots(1, 2, figsize=(6.0, 2.6))
for i, a in enumerate(show):
    s = SWEEP[SWEEP["arm"] == a]
    a1.plot(s["threshold"], s["macro_f1"], "-o", ms=3, lw=1.5, color=SERIES[i], label=ARM_LABELS[a])
s = SWEEP[SWEEP["arm"] == PROPOSED]
a2.plot(s["threshold"], s["attack_detection_rate"], "-o", ms=3, lw=1.5, color=SERIES[3], label="Attack detection rate")
a2.plot(s["threshold"], s["benign_fpr"], "-o", ms=3, lw=1.5, color=SERIES[5], label="Benign false-positive rate")
for ax_ in (a1, a2):
    ax_.axvline(THR, color="#52514e", lw=0.8, ls="--")
    ax_.text(THR, 1.0, f" τ = {THR:.2f} (OOF)", transform=ax_.get_xaxis_transform(),
             fontsize=6.5, color="#52514e", va="bottom")
    ax_.set_xlabel("Gate threshold τ")
a1.set_ylabel("End-to-end macro-F1"); a2.set_ylabel("Rate")
a1.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.25), fontsize=6.5)
a2.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.25), fontsize=6.5)
savefig(f, "fig_threshold_sensitivity")

## 13. Export everything for the paper

In [ ]:
with pd.ExcelWriter(res("paper_tables.xlsx")) as xw:
    for name in ["table_ablation", "table_per_class_f1_all_arms", "table_gate_binary",
                 "table_stage2_isolated", "table_mcnemar", "table_bootstrap_ci",
                 "table_threshold_sensitivity", "table_seed_repeats", "table_seed_summary",
                 "table_class_distribution", "table_feature_selection", "table_train_test_overlap"]:
        p = res(name + ".csv")
        if os.path.exists(p):
            pd.read_csv(p).to_excel(xw, sheet_name=name.replace("table_", "")[:31], index=False)

summary = {
    "preprocessing": PREP_LOG, "minority_classes": MINORITY_NAMES, "gate_threshold": THR,
    "gate_threshold_criterion": THRESHOLD_CRITERION, "scale_pos_weight": SPW,
    "ablation": {a: RESULTS[a]["metrics"] for a in ARMS},
    "environment": ENV,
}
save_json(summary, res("RESULTS_SUMMARY.json"))
zip_path = shutil.make_archive(os.path.join(OUT_DIR, "results"), "zip", RES_DIR)
print("\nAll outputs:", RES_DIR)
print("Results archive:", zip_path)
print(ABL[["configuration", "macro_f1", "minority_macro_f1", "weighted_f1", "mcc",
           "attack_detection_rate", "benign_fpr"]].round(4).to_string())